In [159]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import os

# 1. Data Retrieval

In [160]:
# STEP 1 — API Setup

API_KEY = 'oe_QGgWdwVTERbwcSmN5Nhwnt'

BASE_URL = "https://api.openelectricity.org.au/v4"

NETWORK = "NEM"

STATUS = "operating"

INTERVAL = "5m"

START_DATE = "2026-05-09T00:00:00"

END_DATE = "2026-05-16T00:00:00"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}"
}

In [161]:
# STEP 2 — Get Facilities List

facility_url = f"{BASE_URL}/facilities/"

facility_params = {
    "network_id": NETWORK,
    "status_id": STATUS
}

facility_response = requests.get(
    facility_url,
    headers=HEADERS,
    params=facility_params
)

print("Status Code:", facility_response.status_code)


# Convert JSON

facility_json = facility_response.json()


# Extract data

facility_data = facility_json["data"]


# Check total facilities

print("Total Facilities:", len(facility_data))


# View first facility

facility_data[0]

Status Code: 200
Total Facilities: 432


{'code': 'ADP',
 'code_display': 'ADP',
 'name': 'Adelaide Desalination',
 'network_id': 'NEM',
 'network_region': 'SA1',
 'description': '<p>The Adelaide Desalination plant (ADP), formerly known as the Port Stanvac Desalination Plant, is a sea water reverse osmosis desalination plant located in Lonsdale, South Australia which has the capacity to provide the city of Adelaide with up to 50% of its drinking water needs.</p>',
 'osm_way_id': '12798948',
 'location': {'lat': -35.100751, 'lng': 138.483357},
 'units': [{'code': 'ADPBA1',
   'code_display': 'ADPBA1',
   'fueltech_id': 'battery',
   'status_id': 'operating',
   'capacity_registered': 7.76,
   'capacity_maximum': 6.15,
   'capacity_storage': 12.6,
   'data_first_seen': '2024-09-18T10:40:00+10:00',
   'data_last_seen': '2026-05-17T00:20:00+10:00',
   'dispatch_type': 'BIDIRECTIONAL',
   'commencement_date': '2021-05-17T14:00:00+10:00',
   'commencement_date_specificity': 'day',
   'expected_closure_date': '2040-12-31T14:00:00+10

In [162]:
# STEP 3 — Create Empty Data Dictionary

data_dict = {

    "facility_name": [],

    "facility_code": [],

    "unit_code": [],

    "power_mw": [],

    "emissions_t": [],

    "timestamp": [],

    "lat": [],

    "lon": []
}

In [163]:
# STEP 4 Explore Facility Structure

first_facility = facility_data[0]

first_facility

{'code': 'ADP',
 'code_display': 'ADP',
 'name': 'Adelaide Desalination',
 'network_id': 'NEM',
 'network_region': 'SA1',
 'description': '<p>The Adelaide Desalination plant (ADP), formerly known as the Port Stanvac Desalination Plant, is a sea water reverse osmosis desalination plant located in Lonsdale, South Australia which has the capacity to provide the city of Adelaide with up to 50% of its drinking water needs.</p>',
 'osm_way_id': '12798948',
 'location': {'lat': -35.100751, 'lng': 138.483357},
 'units': [{'code': 'ADPBA1',
   'code_display': 'ADPBA1',
   'fueltech_id': 'battery',
   'status_id': 'operating',
   'capacity_registered': 7.76,
   'capacity_maximum': 6.15,
   'capacity_storage': 12.6,
   'data_first_seen': '2024-09-18T10:40:00+10:00',
   'data_last_seen': '2026-05-17T00:20:00+10:00',
   'dispatch_type': 'BIDIRECTIONAL',
   'commencement_date': '2021-05-17T14:00:00+10:00',
   'commencement_date_specificity': 'day',
   'expected_closure_date': '2040-12-31T14:00:00+10

In [164]:
# STEP 5 — Extract Facility Information

facility_code = first_facility["code"]

facility_name = first_facility["name"]

lat = first_facility["location"]["lat"]

lon = first_facility["location"]["lng"]

print(facility_code)

print(facility_name)

ADP
Adelaide Desalination


In [165]:
# STEP 6 — Request Power + Emissions Data

data_url = f"{BASE_URL}/data/facilities/{NETWORK}"

data_params = {
    "metrics": ["power", "emissions"],
    "interval": INTERVAL,
    "facility_code": facility_code,
    "date_start": START_DATE,
    "date_end": END_DATE
}

data_response = requests.get(
    data_url,
    headers=HEADERS,
    params=data_params
)

print(data_response.status_code)

200


In [166]:
# STEP 7 — Convert JSON
data_json = data_response.json()

data_json.keys()

dict_keys(['version', 'created_at', 'success', 'data'])

In [167]:
# STEP 8 — Extract Power + Emissions

data_list = data_json["data"]

power_data = data_list[0]["results"][0]

emissions_data = data_list[1]["results"][0]

power_data.keys()

dict_keys(['name', 'date_start', 'date_end', 'columns', 'data'])

In [168]:
# First 5 power rows
power_data["data"][:5]

[['2026-05-09T00:00:00+10:00', -0.005],
 ['2026-05-09T00:05:00+10:00', 0.027],
 ['2026-05-09T00:10:00+10:00', 0.008],
 ['2026-05-09T00:15:00+10:00', 0.071],
 ['2026-05-09T00:20:00+10:00', 0.009]]

In [169]:
# STEP 9 Integrate Facility Data

for power_row, emissions_row in zip(
    power_data["data"],
    emissions_data["data"]
):

    timestamp = power_row[0]

    power_value = power_row[1]

    emissions_value = emissions_row[1]


    data_dict["facility_name"].append(
        facility_name
    )

    data_dict["facility_code"].append(
        facility_code
    )

    data_dict["unit_code"].append(
        power_data["columns"]["unit_code"]
    )

    data_dict["power_mw"].append(
        power_value
    )

    data_dict["emissions_t"].append(
        emissions_value
    )

    data_dict["timestamp"].append(
        timestamp
    )

    data_dict["lat"].append(
        lat
    )

    data_dict["lon"].append(
        lng
    )

In [170]:
# STEP 10 — Create DataFrame
df = pd.DataFrame(data_dict)

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.005,0.0,2026-05-09T00:00:00+10:00,-35.100751,138.483357
1,Adelaide Desalination,ADP,ADPBA1,0.027,0.0,2026-05-09T00:05:00+10:00,-35.100751,138.483357
2,Adelaide Desalination,ADP,ADPBA1,0.008,0.0,2026-05-09T00:10:00+10:00,-35.100751,138.483357
3,Adelaide Desalination,ADP,ADPBA1,0.071,0.0,2026-05-09T00:15:00+10:00,-35.100751,138.483357
4,Adelaide Desalination,ADP,ADPBA1,0.009,0.0,2026-05-09T00:20:00+10:00,-35.100751,138.483357


In [171]:
# STEP 11 — Save CSV

file_name = "raw_data.csv"

if os.path.exists(file_name):

    print("CSV file already exists.")

else:

    df.to_csv(
        file_name,
        index=False
    )

CSV file already exists.


# 2. Data Integration and Materialisation/Caching

In [172]:
# STEP 12 — Load Raw Data
df = pd.read_csv("raw_data.csv")

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.005,0.0,2026-05-09T00:00:00+10:00,-35.100751,138.483357
1,Adelaide Desalination,ADP,ADPBA1,0.027,0.0,2026-05-09T00:05:00+10:00,-35.100751,138.483357
2,Adelaide Desalination,ADP,ADPBA1,0.008,0.0,2026-05-09T00:10:00+10:00,-35.100751,138.483357
3,Adelaide Desalination,ADP,ADPBA1,0.071,0.0,2026-05-09T00:15:00+10:00,-35.100751,138.483357
4,Adelaide Desalination,ADP,ADPBA1,0.009,0.0,2026-05-09T00:20:00+10:00,-35.100751,138.483357


In [173]:
# STEP 13 — Basic Preprocessing
# Remove missing values
df = df.dropna()

# Remove duplicate rows
df = df.drop_duplicates()

# Convert timestamp datatype
df["timestamp"] = pd.to_datetime(
    df["timestamp"]
)

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.005,0.0,2026-05-09 00:00:00+10:00,-35.100751,138.483357
1,Adelaide Desalination,ADP,ADPBA1,0.027,0.0,2026-05-09 00:05:00+10:00,-35.100751,138.483357
2,Adelaide Desalination,ADP,ADPBA1,0.008,0.0,2026-05-09 00:10:00+10:00,-35.100751,138.483357
3,Adelaide Desalination,ADP,ADPBA1,0.071,0.0,2026-05-09 00:15:00+10:00,-35.100751,138.483357
4,Adelaide Desalination,ADP,ADPBA1,0.009,0.0,2026-05-09 00:20:00+10:00,-35.100751,138.483357


In [174]:
# STEP 14 — Sort Data by Time
df = df.sort_values(
    by="timestamp"
)

df = df.reset_index(drop=True)

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.005,0.0,2026-05-09 00:00:00+10:00,-35.100751,138.483357
1,Adelaide Desalination,ADP,ADPBA1,0.027,0.0,2026-05-09 00:05:00+10:00,-35.100751,138.483357
2,Adelaide Desalination,ADP,ADPBA1,0.008,0.0,2026-05-09 00:10:00+10:00,-35.100751,138.483357
3,Adelaide Desalination,ADP,ADPBA1,0.071,0.0,2026-05-09 00:15:00+10:00,-35.100751,138.483357
4,Adelaide Desalination,ADP,ADPBA1,0.009,0.0,2026-05-09 00:20:00+10:00,-35.100751,138.483357


In [175]:
# STEP 15 — Materialisation / Caching

file_name = "integrated_power_emissions_data.csv"

if os.path.exists(file_name):

    print("CSV file already exists.")

else:

    df.to_csv(
        file_name,
        index=False
    )

    print("Integrated CSV saved successfully.")

CSV file already exists.
